# AC-DC Converters

This notebook acts as a compact textbook chapter for diode and thyristor rectifiers. The goal is to connect source waveforms, conduction intervals, and load-current shaping in one place.

## Theory Summary

### Single-phase half-wave rectifier

For an ideal diode feeding an `R` load,

$$v_o(\omega t)=\begin{cases}V_m\sin \omega t,&0\le \omega t\le \pi\\0,&\pi<\omega t<2\pi\end{cases}$$

with

$$V_{dc}=\frac{V_m}{\pi}, \qquad V_{rms}=\frac{V_m}{2}.$$

For a controlled SCR half-wave rectifier with firing angle `\alpha` and resistive load, conduction starts at `\omega t=\alpha` and ends at `\pi`:

$$V_{dc}=\frac{V_m}{2\pi}(1+\cos\alpha).$$

### Single-phase full-wave bridge

An uncontrolled bridge produces

$$v_o(\omega t)=|V_m\sin \omega t|, \qquad V_{dc}=\frac{2V_m}{\pi}, \qquad V_{rms}=\frac{V_m}{\sqrt{2}}.$$

For a fully controlled bridge with continuous current,

$$V_{dc}=\frac{2V_m}{\pi}\cos\alpha.$$

Each thyristor pair conducts for `\pi` radians, so delaying the firing instant shifts the source segment applied to the load.

### Three-phase rectifiers

A three-phase half-wave rectifier always connects the phase having the highest instantaneous phase-to-neutral voltage. The uncontrolled average output is

$$V_{dc}=\frac{3\sqrt{3}}{2\pi}V_{m,ph}=1.17V_{ph,rms}.$$

A six-pulse diode bridge always applies the largest line-to-line voltage to the load:

$$V_{dc}=\frac{3\sqrt{3}}{\pi}V_{m,ph}=1.35V_{LL,rms}.$$

For a fully controlled three-phase bridge with continuous current,

$$V_{dc}=1.35V_{LL,rms}\cos\alpha.$$

The dominant idea is that commutation occurs every `60^\circ`, while each device conducts for `120^\circ` in the six-pulse bridge.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from power_sim import ac_dc, dc_ac, dc_dc, loads, plotter, utils

plt.rcParams["figure.figsize"] = (11, 8)
plt.rcParams["axes.grid"] = True

In [ ]:
f = 50.0
vm = 325.0
alpha = np.deg2rad(60.0)
t = utils.timebase(fundamental_frequency=f, cycles=2.0, samples_per_cycle=1200)
omega_t = 2 * np.pi * f * t

bridge = ac_dc.single_phase_full_bridge_rectifier(
    t,
    source_peak=vm,
    frequency=f,
    alpha=alpha,
    controlled=True,
    continuous_current=True,
)
rl = loads.RLLoad(resistance=8.0, inductance=0.05, allow_negative_current=False).simulate(
    t,
    bridge["output_voltage"],
    settle_cycles=4,
)

fig, axes = plotter.plot_stacked_waveforms(
    omega_t,
    [
        {"y": bridge["source_voltage"], "label": "Source voltage", "ylabel": r"$v_s$ (V)", "color": "tab:blue"},
        {"y": bridge["output_voltage"], "label": "Bridge output", "ylabel": r"$v_o$ (V)", "color": "tab:red"},
        {"y": rl.current, "label": "RL load current", "ylabel": r"$i_o$ (A)", "color": "tab:green"},
    ],
    xlabel=r"$\omega t$ (rad)",
    title="Single-phase fully controlled bridge with RL load",
)
plt.show()

print("Numerical bridge metrics:", bridge["metrics"])
print("RL load average current:", round(rl.average_current, 3), "A")
print("RL load RMS current:", round(rl.rms_current, 3), "A")

## Conduction Interpretation

The code above separates two ideas:

1. `power_sim.ac_dc` decides which source segment is connected to the load.
2. `power_sim.loads` translates that applied voltage into current for `R`, `RL`, `RLE`, or ideal current-source loads.

This mirrors classroom analysis. The converter sets the voltage waveform, and the load differential equation sets the current waveform.

In [ ]:
three_phase = ac_dc.three_phase_bridge_rectifier(
    t,
    phase_peak=325.0,
    frequency=f,
    alpha=np.deg2rad(25.0),
    controlled=True,
)
motor_load = loads.RLELoad(resistance=2.0, inductance=0.03, back_emf=110.0, allow_negative_current=False).simulate(
    t,
    three_phase["output_voltage"],
    settle_cycles=6,
)

fig, axes = plotter.plot_stacked_waveforms(
    omega_t,
    [
        {"y": three_phase["phase_voltages"][0], "label": "Phase a voltage", "ylabel": r"$v_a$ (V)", "color": "tab:blue"},
        {"y": three_phase["output_voltage"], "label": "Bridge output", "ylabel": r"$v_d$ (V)", "color": "tab:red"},
        {"y": motor_load.current, "label": "RLE load current", "ylabel": r"$i_d$ (A)", "color": "tab:purple"},
    ],
    xlabel=r"$\omega t$ (rad)",
    title="Three-phase controlled bridge feeding an RLE load",
)
plt.show()

print("Three-phase bridge average output voltage:", round(three_phase["metrics"]["average"], 3), "V")
print("Average motor current:", round(motor_load.average_current, 3), "A")